# Proof of Concept

Calculate a simple forward pass from 3DGS to 3DGS

In [35]:
import os
os.environ["UCO3D_DATASET_ROOT"] = "./data/uco3d_data_raw/"

from uco3d.dataset_utils.utils import get_dataset_root
from uco3d import UCO3DDataset, UCO3DFrameDataBuilder



dataset_root = get_dataset_root(assert_exists=True)
subset_lists_file = os.path.join(
    dataset_root,
    "set_lists",
    "set_lists_all-categories.sqlite",
)

official_dataset = UCO3DDataset(
    subset_lists_file=subset_lists_file,
    subsets=["train"],
    frame_data_builder=UCO3DFrameDataBuilder(
        apply_alignment=True,
        load_images=True,
        load_depths=False,
        load_masks=True,
        load_depth_masks=True,
        load_gaussian_splats=True,
        gaussian_splats_truncate_background=True,
        load_point_clouds=True,
        load_segmented_point_clouds=True,
        load_sparse_point_clouds=True,
        box_crop=True,
        box_crop_context=0.4,
        load_frames_from_videos=True,
        image_height=800,
        image_width=800,
        undistort_loaded_blobs=True,
    )
)

frame_data = official_dataset[0]

print(official_dataset)

UCO3DDataset #frames=9399


In [36]:
from uco3d.dataset_utils.data_types import GaussianSplats
from torch import nn
import torch

class POC_Network(nn.Module):
    def __init__(self, dim, bottleneck_size):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(dim, 32),
            nn.ReLU(),
            nn.Linear(32, bottleneck_size),
            nn.ReLU(),
            nn.Linear(bottleneck_size, 32),
            nn.ReLU(),
            nn.Linear(32, dim)
        )

    def forward(self, x):
        x = self.model(x)
        return x

    def reconstruct_splats(self, splats):
        means = splats.means  # (N, 3)
        sh0 = splats.sh0     # (N, 3)
        scales = splats.scales  # (N, 3)
        quats = splats.quats    # (N, 4)
        opacities = splats.opacities  # (N,)

        flat = torch.cat([means, sh0, scales, quats, opacities.unsqueeze(-1)], dim=-1)

        reconstructed_flat = self.model(flat)
        means_recon = reconstructed_flat[:, 0:3]
        sh0_recon = reconstructed_flat[:, 3:6]
        scales_recon = reconstructed_flat[:, 6:9]
        quats_recon = reconstructed_flat[:, 9:13]
        opacities_recon = reconstructed_flat[:, 13:14].squeeze(-1)

        return GaussianSplats(
            means=means_recon,
            sh0=sh0_recon,
            scales=scales_recon,
            quats=quats_recon,
            opacities=opacities_recon,
            shN=splats.shN,
            fg_mask=splats.fg_mask,
        )

In [37]:
import wandb
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader


EPOCHS = 1000
LR = 1e-3
BOTTLENECK_SIZES = [10]
SAMPLES_LIST = [1, 2, 5, 10]

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

def train_run(bottleneck_size, num_samples):
    # Finish any live run and restart the service if it crashed (MailboxClosedError)
    try:
        wandb.finish()
    except Exception:
        pass
    try:
        wandb.teardown()
    except Exception:
        pass

    network = POC_Network(dim=14, bottleneck_size=bottleneck_size).to(DEVICE)
    loader = DataLoader(official_dataset, batch_size=1, shuffle=False)
    optim = AdamW(network.parameters(), lr=LR)

    run = wandb.init(
        entity="leonardkuech-technical-university-of-munich",
        project="feed-forward radiance field conversion",
        config={
            "learning_rate": LR,
            "bottleneck_size": bottleneck_size,
            "dataset": "uco3d",
            "epochs": EPOCHS,
            "samples": num_samples,
            "device": str(DEVICE),
        },
        reinit="finish_previous",
    )

    try:
        for epoch in range(EPOCHS):
            optim.zero_grad()
            batch = next(iter(loader))
            x = batch[0]["splat_tensor"].to(DEVICE)
            output = network(x)
            loss = F.mse_loss(x, output)
            loss.backward()
            optim.step()
            run.log({"loss": loss.item()})
    finally:
        wandb.finish()


In [38]:
for bottleneck_size in BOTTLENECK_SIZES:
    for num_samples in SAMPLES_LIST:
        print(f"Training: bottleneck={bottleneck_size}, samples={num_samples}")
        train_run(bottleneck_size, num_samples)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/leonardkuch/.netrc.


Training: bottleneck=10, samples=1
wandb: 
wandb: Find logs at: wandb/run-20260506_130649-8y6fqb8l/logs


TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'NoneType'>